# STAGE 13B.4 — Historical Environmental Feature Extraction for Replacement Controls

**Malaysia-only landslide FYP | Conservatively reproduces Stage 05B extraction**

**Objective:** Extract historical rainfall, soil moisture, land cover and current waterway distance for Stage 13B.3's shortlisted control candidates, without approving replacements or modifying the existing training set.

**Important controls:** Reference-day-exclusive rainfall and moisture windows; explicit Earth Engine collection-count checks; source/error audit; HIGH land-cover reliability requirement; no candidate is accepted until Stage 13B.5 checks spatial exclusions, uniqueness, matched-group identity and complete QC. Annual historical land-cover composites are not guaranteed to be strictly pre-event.

Run in VS Code / Jupyter with access to your existing `C:\Landslide_FYP` project and authenticated Google Earth Engine.

## PART A — Environment and input verification
### 1. Imports and configuration

In [6]:
# Uncomment if packages are not yet installed:
# %pip install earthengine-api pandas numpy geopandas rasterio pyproj shapely

from pathlib import Path
from collections import defaultdict
import json
import re
import time
import warnings

import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
import geopandas as gpd
import ee

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
EE_PROJECT = "fyp-landslide-malaysia"  # Same as original Stage 05B
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"
RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"

STAGE13B3_DIR = DEV_DIR / "05_model_results" / "stage13b3_control_recovery"
MANIFEST_PATH = STAGE13B3_DIR / "candidate_temporal_extraction_manifest.csv"
LANDCOVER_ROOT = RAW_DIR / "03_Factors" / "Landcover" / "GLC_FCS30D"
WATERWAYS_ROOT = RAW_DIR / "03_Factors" / "Waterways"
OUTPUT_DIR = DEV_DIR / "05_model_results" / "stage13b4_historical_control_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep method/scale consistent with original Stage 05B.
RAIN_DAYS = (1, 3, 7, 30)
EE_SAMPLING_SCALE_M = 11132
WATERWAY_DISTANCE_CRS = "EPSG:3857"
EE_CHUNK_ROWS = 35
PAUSE_SECONDS = 0.25

# Only for explicit resume after inspecting saved results. Set to True for first run.
EXTRACT_EE = True
EXTRACT_LOCAL_WATERWAYS = True

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Stage 13B.3 manifest missing: {MANIFEST_PATH}")

print("Manifest:", MANIFEST_PATH)
print("Outputs:", OUTPUT_DIR)
print("WARNING: This notebook does not approve or assign control replacements.")

Manifest: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b3_control_recovery\candidate_temporal_extraction_manifest.csv
Outputs: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b4_historical_control_features


### 2. Load manifest, validate IDs, and de-duplicate repeated candidate/date pairs

In [7]:
manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
required = {
    "matched_positive_id", "old_control_sample_id", "sample_id",
    "longitude", "latitude", "reference_date", "candidate_rank",
}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest missing columns: {sorted(missing)}")

manifest["reference_date"] = pd.to_datetime(manifest["reference_date"], errors="coerce")
for col in ("longitude", "latitude"):
    manifest[col] = pd.to_numeric(manifest[col], errors="coerce")

if manifest[list(required - {"candidate_rank"})].isna().any().any():
    raise ValueError("Manifest has null IDs, dates or coordinates; inspect Stage13B.3 output.")
if not manifest["longitude"].between(99, 120).all() or not manifest["latitude"].between(-2, 9).all():
    raise ValueError("Some coordinates are outside broad Malaysian geographic bounds.")

# A background candidate may be shortlisted for different groups/dates.
# The extraction key is location ID + exact date, not only sample ID.
key_cols = ["sample_id", "reference_date"]
coords_by_id = manifest.groupby("sample_id")[ ["longitude", "latitude"] ].nunique()
if (coords_by_id > 1).any().any():
    raise ValueError("A sample_id has inconsistent coordinates. Do not extract until corrected.")

unique_sites = manifest.drop_duplicates(key_cols).copy().reset_index(drop=True)
unique_sites["extract_row_id"] = np.arange(len(unique_sites), dtype=int)
unique_sites["reference_year"] = unique_sites["reference_date"].dt.year.astype(int)
unique_sites["reference_date_str"] = unique_sites["reference_date"].dt.strftime("%Y-%m-%d")

print("Shortlisted slot-candidate rows:", len(manifest))
print("Unique candidate/date observations:", len(unique_sites))
print("Unique locations:", unique_sites["sample_id"].nunique())
print("Historical reference years:", sorted(unique_sites["reference_year"].unique()))
unique_sites.to_csv(OUTPUT_DIR / "unique_candidate_date_extraction_keys.csv", index=False)
display(unique_sites.head(8))

Shortlisted slot-candidate rows: 170
Unique candidate/date observations: 154
Unique locations: 72
Historical reference years: [np.int64(2008), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2021)]


,matched_positive_id,old_control_sample_id,reference_date,sample_id,longitude,latitude,distance_to_case_km,candidate_rank,candidate_status,temporal_role,...,reference_month,reference_day,extract_rainfall,extract_era5_soil_moisture,extract_historical_landcover,verify_waterway_distance,verify_original_pseudoabsence_spatial_exclusion,verified_replacement,extract_row_id,reference_date_str
0,POS_0008,BG_01201,2011-11-25,BG_00263,101.178150,4.340972,4.341592,1,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,11,25,True,True,True,True,True,False,0,2011-11-25
1,POS_0008,BG_01201,2011-11-25,BG_00233,101.193053,4.270633,6.479220,2,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,11,25,True,True,True,True,True,False,1,2011-11-25
2,POS_0008,BG_01201,2011-11-25,BG_00265,101.212613,4.335245,7.210901,3,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,11,25,True,True,True,True,True,False,2,2011-11-25
3,POS_0008,BG_01201,2011-11-25,BG_00269,101.051497,4.359263,12.310573,4,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,11,25,True,True,True,True,True,False,3,2011-11-25
4,POS_0008,BG_01201,2011-11-25,BG_00267,101.157916,4.433945,13.576062,5,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,11,25,True,True,True,True,True,False,4,2011-11-25
5,POS_0013,BG_00761,2016-07-22,BG_00360,101.735466,2.983178,14.087608,1,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,7,22,True,True,True,True,True,False,5,2016-07-22
6,POS_0013,BG_00761,2016-07-22,BG_00366,101.891537,3.039185,16.893096,2,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,7,22,True,True,True,True,True,False,6,2016-07-22
7,POS_0013,BG_00761,2016-07-22,BG_00391,101.914930,3.187754,19.738718,3,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...,matched_control_candidate,...,7,22,True,True,True,True,True,False,7,2016-07-22


## PART B — Historical land cover
### 3. Index local GLC_FCS30D rasters and sample in the original order

In [8]:
year_index = defaultdict(list)
if LANDCOVER_ROOT.exists():
    for path in sorted(LANDCOVER_ROOT.rglob("*.tif")):
        years = re.findall(r"(19\d{2}|20\d{2})", path.name)
        for year in set(years):
            year_index[int(year)].append(path)

print("GLC raster years:", sorted(year_index))

# Return a value only for a source-valid pixel. Do not interpret raster nodata as land cover.
def local_pixel_value(path, lon, lat):
    with rasterio.open(path) as src:
        if src.crs is None:
            raise ValueError(f"No CRS in raster: {path}")
        x, y = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True).transform(lon, lat)
        bounds = src.bounds
        if not (bounds.left <= x < bounds.right and bounds.bottom <= y < bounds.top):
            return np.nan
        sample = next(src.sample([(x, y)], masked=True))[0]
        if np.ma.is_masked(sample):
            return np.nan
        val = float(sample)
        if not np.isfinite(val) or (src.nodata is not None and val == src.nodata):
            return np.nan
        return val

local_rows = []
for row in unique_sites.itertuples(index=False):
    raw, sourcefile, error = np.nan, "", ""
    for path in year_index.get(row.reference_year, []):
        try:
            raw = local_pixel_value(path, row.longitude, row.latitude)
            if pd.notna(raw):
                sourcefile = str(path.relative_to(LANDCOVER_ROOT))
                break
        except Exception as exc:
            error = str(exc)[:240]
    local_rows.append({
        "extract_row_id": row.extract_row_id,
        "landcover_code_raw": raw,
        "landcover_source": "GLC_FCS30D" if pd.notna(raw) else None,
        "landcover_source_file": sourcefile,
        "landcover_status": "ok" if pd.notna(raw) else "local_year_missing_or_no_valid_pixel",
        "landcover_local_error": error,
    })
local_df = pd.DataFrame(local_rows)
unique_sites = unique_sites.merge(local_df, on="extract_row_id", validate="one_to_one")
print(unique_sites["landcover_status"].value_counts(dropna=False).to_string())

GLC raster years: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]
landcover_status
local_year_missing_or_no_valid_pixel    109
ok                                       45


### 4. Authenticate Google Earth Engine and check collection availability

In [9]:
# Earth Engine is mandatory for rainfall and soil moisture extraction.
# Log in if needed: run ee.Authenticate() once in an interactive cell.
try:
    ee.Initialize(project=EE_PROJECT)
except Exception as exc:
    raise RuntimeError(
        "Earth Engine initialization failed. Run ee.Authenticate() interactively, "
        "confirm EE_PROJECT, and retry. Original error: " + str(exc)
    )

GPM_COLLECTION = "NASA/GPM_L3/IMERG_V07"
ERA5_COLLECTION = "ECMWF/ERA5_LAND/DAILY_AGGR"
DYNAMIC_WORLD_COLLECTION = "GOOGLE/DYNAMICWORLD/V1"
SOIL_BANDS = [
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
]

# Check temporal collection availability before using summed images.
# Without this, an empty collection could produce invalid or misleading results.
collection_audit = []
for date_str in sorted(unique_sites["reference_date_str"].unique()):
    end = ee.Date(date_str)
    counts = {}
    for days in RAIN_DAYS:
        counts[f"gpm_{days}d_image_count"] = ee.ImageCollection(GPM_COLLECTION).filterDate(
            end.advance(-days, "day"), end
        ).size().getInfo()
    counts["era5_7d_image_count"] = ee.ImageCollection(ERA5_COLLECTION).filterDate(
        end.advance(-7, "day"), end
    ).size().getInfo()
    collection_audit.append({"reference_date": date_str, **counts})

counts_df = pd.DataFrame(collection_audit)
counts_df.to_csv(OUTPUT_DIR / "earth_engine_collection_counts.csv", index=False)
print("Date groups:", len(counts_df))
print("Dates with absent 30-day rainfall:", int(counts_df["gpm_30d_image_count"].eq(0).sum()))
print("Dates with absent ERA5 window:", int(counts_df["era5_7d_image_count"].eq(0).sum()))
display(counts_df.head())

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Date groups: 30
Dates with absent 30-day rainfall: 0
Dates with absent ERA5 window: 0


,reference_date,gpm_1d_image_count,gpm_3d_image_count,gpm_7d_image_count,gpm_30d_image_count,era5_7d_image_count
0,2008-10-04,48,144,336,1440,7
1,2011-01-29,48,144,336,1440,7
2,2011-11-25,48,144,336,1440,7
3,2012-11-15,48,144,336,1440,7
4,2013-10-23,48,144,336,1440,7


### 5. Dynamic World annual-mode fallback for local GLC gaps

In [10]:
# Dynamic World is used only when the local historical GLC pixel is missing.
# It is not used before collection availability; 2015 partial-year is LOW reliability.
# Annual mode may use post-event imagery; record as temporal limitation.

def ee_reduce_points(image, frame, names, scale=EE_SAMPLING_SCALE_M):
    """Batch point sample with row IDs, returning empty/missing values transparently."""
    if frame.empty:
        return pd.DataFrame(columns=["extract_row_id", *names])
    out = []
    for first in range(0, len(frame), EE_CHUNK_ROWS):
        part = frame.iloc[first:first+EE_CHUNK_ROWS]
        points = [
            ee.Feature(ee.Geometry.Point([float(r.longitude), float(r.latitude)]),
                       {"extract_row_id": int(r.extract_row_id)})
            for r in part.itertuples(index=False)
        ]
        result = image.reduceRegions(
            collection=ee.FeatureCollection(points),
            reducer=ee.Reducer.first(),
            scale=scale,
        ).getInfo()
        for feature in result.get("features", []):
            prop = feature.get("properties", {})
            out.append({"extract_row_id": prop.get("extract_row_id"),
                        **{name: prop.get(name) for name in names}})
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(out, columns=["extract_row_id", *names])

fallback = unique_sites.loc[unique_sites["landcover_code_raw"].isna()].copy()
dw_rows = []
for year, year_df in fallback.groupby("reference_year"):
    start = ee.Date.fromYMD(int(year), 1, 1)
    col = ee.ImageCollection(DYNAMIC_WORLD_COLLECTION).filterDate(start, start.advance(1, "year"))
    count = col.size().getInfo()
    if count == 0:
        dw_rows.extend({"extract_row_id": int(v), "dw_label": np.nan, "dw_count": 0}
                       for v in year_df["extract_row_id"])
        continue
    image = col.select("label").mode().rename("dw_label")
    sampled = ee_reduce_points(image, year_df, ["dw_label"], scale=10)
    sampled["dw_count"] = int(count)
    dw_rows.extend(sampled.to_dict("records"))
    print(f"Dynamic World {year}: {len(sampled)} points; {count} collection scenes")

# ============================================================
# FIX: Dynamic World fallback and Pandas dtype compatibility
# ============================================================

# 1. Convert Dynamic World results into a DataFrame
dw_df = pd.DataFrame(
    dw_rows,
    columns=["extract_row_id", "dw_label", "dw_count"]
)

# 2. Merge results with the original extraction records
unique_sites = unique_sites.merge(
    dw_df,
    on="extract_row_id",
    how="left",
    validate="one_to_one"
)

# 3. Ensure categorical columns support text values
for col in ["landcover_source", "landcover_class"]:
    if col not in unique_sites.columns:
        unique_sites[col] = pd.Series(
            pd.NA,
            index=unique_sites.index,
            dtype="object"
        )
    else:
        unique_sites[col] = unique_sites[col].astype("object")

# 4. Identify records requiring Dynamic World fallback
use_dw = (
    unique_sites["landcover_code_raw"].isna()
    & unique_sites["dw_label"].notna()
)

# 5. Apply fallback only when eligible records exist
if use_dw.any():

    unique_sites.loc[
        use_dw,
        "landcover_code_raw"
    ] = unique_sites.loc[
        use_dw,
        "dw_label"
    ]

    unique_sites.loc[
        use_dw,
        "landcover_source"
    ] = "Dynamic World"

    unique_sites.loc[
        use_dw,
        "landcover_class"
    ] = "lc_fallback_dynamic_world"

else:
    print("No records require Dynamic World fallback.")

# 6. Assign the reference year
unique_sites["landcover_year"] = unique_sites["reference_year"]

# 7. Verify output
print("\nDynamic World fallback records:", int(use_dw.sum()))

print("\nLand cover source distribution:")
print(unique_sites["landcover_source"].value_counts(dropna=False))

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Dynamic World 2016: 30 points; 1141292 collection scenes


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Dynamic World 2017: 40 points; 954172 collection scenes


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Dynamic World 2018: 5 points; 1913534 collection scenes


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Dynamic World 2021: 34 points; 1737226 collection scenes
No records require Dynamic World fallback.

Dynamic World fallback records: 0

Land cover source distribution:
landcover_source
NaN           109
GLC_FCS30D     45
Name: count, dtype: int64


## PART C — Antecedent rainfall and soil moisture
### 6. Extract identical GPM V07 and ERA5-Land predictors used by Stage 05B

In [11]:
def gpm_image(date_str, days):
    end = ee.Date(date_str)
    # Equivalent to original Stage 05B: precipitation mm/hr x 0.5h image timestep.
    return ee.ImageCollection(GPM_COLLECTION).filterDate(
        end.advance(-int(days), "day"), end
    ).select("precipitation").map(lambda img: img.multiply(0.5)).sum().rename(f"rain_{days}d_mm")

def era5_image(date_str):
    end = ee.Date(date_str)
    return ee.ImageCollection(ERA5_COLLECTION).filterDate(
        end.advance(-7, "day"), end
    ).select(SOIL_BANDS).mean().rename([
        "soil_moisture_0_7cm_7d_mean",
        "soil_moisture_7_28cm_7d_mean",
    ])

rain_cols = [f"rain_{d}d_mm" for d in RAIN_DAYS]
sm_cols = ["soil_moisture_0_7cm_7d_mean", "soil_moisture_7_28cm_7d_mean"]
climate_rows = []
climate_errors = []
for date_str, subset in unique_sites.groupby("reference_date_str", sort=True):
    qc = counts_df.loc[counts_df["reference_date"].eq(date_str)].iloc[0]
    available_rain = [d for d in RAIN_DAYS if int(qc[f"gpm_{d}d_image_count"]) > 0]
    valid_era5 = int(qc["era5_7d_image_count"]) > 0
    # Important: reject an absent source rather than accidentally summing an empty collection.
    images = [gpm_image(date_str, days) for days in available_rain]
    if valid_era5:
        images.append(era5_image(date_str))
    try:
        if images:
            sampled = ee_reduce_points(ee.Image.cat(images), subset,
                [f"rain_{d}d_mm" for d in available_rain] + (sm_cols if valid_era5 else []))
        else:
            sampled = pd.DataFrame({"extract_row_id": subset["extract_row_id"]})
        for col in rain_cols + sm_cols:
            if col not in sampled.columns:
                sampled[col] = np.nan
        climate_rows.extend(sampled[["extract_row_id", *rain_cols, *sm_cols]].to_dict("records"))
        print("Climate:", date_str, "candidate/date rows:", len(subset))
    except Exception as exc:
        # Batch-level failure remains missing, never silently marked successful.
        climate_errors.append({"reference_date": date_str, "error": repr(exc)[:500]})
        for row_id in subset["extract_row_id"]:
            climate_rows.append({"extract_row_id": row_id, **{name: np.nan for name in rain_cols + sm_cols}})
    time.sleep(PAUSE_SECONDS)

climate_df = pd.DataFrame(climate_rows)
if climate_df["extract_row_id"].duplicated().any():
    raise ValueError("Climate batch returned duplicated extraction row IDs")
unique_sites = unique_sites.merge(climate_df, on="extract_row_id", validate="one_to_one")
pd.DataFrame(climate_errors, columns=["reference_date", "error"]).to_csv(
    OUTPUT_DIR / "earth_engine_extraction_errors.csv", index=False)
print("Climate extraction errors:", len(climate_errors))

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Climate: 2008-10-04 candidate/date rows: 5


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Climate: 2011-01-29 candidate/date rows: 5


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


...

Climate: 2021-12-12 candidate/date rows: 5


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Climate: 2021-12-18 candidate/date rows: 8


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Climate: 2021-12-19 candidate/date rows: 6
Climate extraction errors: 0


### 7. Current OSM waterway distance (same EPSG:3857 implementation as Stage 05B)

In [12]:
waterway_files = []
if WATERWAYS_ROOT.exists():
    waterway_files = sorted(path for path in WATERWAYS_ROOT.rglob("*")
                            if path.suffix.lower() in {".shp", ".gpkg", ".geojson"})

water_parts = []
water_errors = []
for path in waterway_files:
    try:
        segment = gpd.read_file(path)
        if segment.empty or segment.crs is None:
            continue
        segment = segment.to_crs("EPSG:4326")
        segment = segment.loc[segment.geometry.geom_type.isin(["LineString", "MultiLineString"])]
        if not segment.empty:
            water_parts.append(segment[["geometry"]].copy())
    except Exception as exc:
        water_errors.append({"file": str(path), "error": repr(exc)[:350]})

unique_sites["distance_waterway_current_m"] = np.nan
unique_sites["waterway_source"] = pd.NA
if water_parts:
    water = gpd.GeoDataFrame(pd.concat(water_parts, ignore_index=True),
                             geometry="geometry", crs="EPSG:4326").to_crs(WATERWAY_DISTANCE_CRS)
    water_union = water.geometry.union_all() if hasattr(water.geometry, "union_all") else water.geometry.unary_union
    points = gpd.GeoSeries(gpd.points_from_xy(unique_sites["longitude"], unique_sites["latitude"]),
                           crs="EPSG:4326").to_crs(WATERWAY_DISTANCE_CRS)
    unique_sites["distance_waterway_current_m"] = points.distance(water_union).to_numpy()
    unique_sites["waterway_source"] = "Current_Geofabrik_OSM"
else:
    print("WARNING: No valid waterways; all distances remain missing")

pd.DataFrame(water_errors, columns=["file", "error"]).to_csv(
    OUTPUT_DIR / "waterway_source_errors.csv", index=False)
print("Waterway vectors loaded:", len(water_parts))
print("Waterway distances available:", unique_sites["distance_waterway_current_m"].notna().sum())

Waterway vectors loaded: 1
Waterway distances available: 154


## PART D — Reproduce Stage 05B-QC classification and readiness
### 8. Harmonize GLC and Dynamic World raw classes using the *original* mapping

In [13]:
HARMONISED_NAME_TO_CODE = {
    "FOREST": 1, "SHRUB": 2, "GRASS": 3, "CROPLAND": 4,
    "BUILT": 5, "WETLAND": 6, "BARE": 7, "WATER": 8, "OTHER": 9,
}
GLC_GROUPS = {
    "CROPLAND": {10, 11, 12, 20},
    "FOREST": {51, 52, 61, 62, 71, 72, 81, 82, 91, 92},
    "SHRUB": {120, 121, 122}, "GRASS": {130},
    "WETLAND": {181, 182, 183, 184, 185, 186, 187},
    "BUILT": {190}, "BARE": {150, 152, 153, 200, 201, 202},
    "WATER": {210}, "OTHER": {140, 220},
}
DW_MAP = {0: "WATER", 1: "FOREST", 2: "GRASS", 3: "WETLAND",
          4: "CROPLAND", 5: "SHRUB", 6: "BUILT", 7: "BARE", 8: "OTHER"}

def harmonise_landcover(source, raw_code):
    if pd.isna(source) or pd.isna(raw_code):
        return pd.NA, np.nan
    try:
        code = int(round(float(raw_code)))
    except (ValueError, TypeError):
        return pd.NA, np.nan
    if source == "GLC_FCS30D":
        if code in {0, 250}:
            return pd.NA, np.nan
        for category, codes in GLC_GROUPS.items():
            if code in codes:
                return category, HARMONISED_NAME_TO_CODE[category]
    if source == "Dynamic_World":
        category = DW_MAP.get(code)
        if category is not None:
            return category, HARMONISED_NAME_TO_CODE[category]
    return pd.NA, np.nan

mapped = [harmonise_landcover(a, b) for a, b in zip(
    unique_sites["landcover_source"], unique_sites["landcover_code_raw"])]
unique_sites["landcover_harmonised"] = [a for a, _ in mapped]
unique_sites["landcover_harmonised_code"] = [b for _, b in mapped]
unique_sites["landcover_reliability"] = np.select([
    unique_sites["landcover_harmonised"].isna(),
    unique_sites["landcover_source"].eq("GLC_FCS30D"),
    unique_sites["landcover_source"].eq("Dynamic_World") & unique_sites["reference_year"].eq(2015),
    unique_sites["landcover_source"].eq("Dynamic_World"),
], ["MISSING", "HIGH", "LOW_PARTIAL_YEAR", "HIGH"], default="UNKNOWN")
unique_sites["water_background_invalid"] = unique_sites["landcover_harmonised"].eq("WATER")
unique_sites["rainfall_complete"] = unique_sites[rain_cols].notna().all(axis=1)
unique_sites["rainfall_logical"] = unique_sites[rain_cols].apply(
    pd.to_numeric, errors="coerce").ge(0).all(axis=1)
unique_sites["rainfall_valid"] = unique_sites["rainfall_complete"] & unique_sites["rainfall_logical"]
unique_sites["soil_moisture_complete"] = unique_sites[sm_cols].notna().all(axis=1)
unique_sites["landcover_valid_strict"] = (
    unique_sites["landcover_harmonised"].notna()
    & unique_sites["landcover_reliability"].eq("HIGH")
    & ~unique_sites["water_background_invalid"]
)
unique_sites["waterway_available"] = unique_sites["distance_waterway_current_m"].notna()
unique_sites["full_temporal_row_valid"] = (
    unique_sites["rainfall_valid"]
    & unique_sites["soil_moisture_complete"]
    & unique_sites["landcover_valid_strict"]
    & unique_sites["waterway_available"]
)

print("Rows passing original feature QC:", int(unique_sites["full_temporal_row_valid"].sum()), "/", len(unique_sites))
print("IMPORTANT: Row QC is only a preliminary gate; no candidate is an approved control.")

Rows passing original feature QC: 43 / 154
IMPORTANT: Row QC is only a preliminary gate; no candidate is an approved control.


### 9. Replicate extracted features back to each candidate/slot occurrence

In [14]:
# Multiple target cases can shortlist the same candidate/date pair.
# Join on BOTH background sample ID and reference date; never only sample ID.
feature_cols = [
    "sample_id", "reference_date", "landcover_code_raw", "landcover_source",
    "landcover_source_file", "landcover_status", "landcover_year",
    "landcover_harmonised", "landcover_harmonised_code", "landcover_reliability",
    "rain_1d_mm", "rain_3d_mm", "rain_7d_mm", "rain_30d_mm",
    "soil_moisture_0_7cm_7d_mean", "soil_moisture_7_28cm_7d_mean",
    "distance_waterway_current_m", "waterway_source",
    "rainfall_valid", "soil_moisture_complete", "landcover_valid_strict",
    "water_background_invalid", "waterway_available", "full_temporal_row_valid",
]
slot_features = manifest.merge(unique_sites[feature_cols],
                              on=["sample_id", "reference_date"],
                              how="left", validate="many_to_one")
if len(slot_features) != len(manifest):
    raise AssertionError("Candidate slot count changed in merge")
slot_features["spatial_exclusion_verified"] = False
slot_features["control_assignment_verified"] = False
slot_features["verified_replacement"] = False
slot_features["temporal_pre_event_validation_status"] = "annual_landcover_composite_not_independently_pre_event_verified"
slot_features["candidate_status"] = np.where(
    slot_features["full_temporal_row_valid"].fillna(False),
    "FEATURE_QC_PASS_SPATIAL_AND_ASSIGNMENT_PENDING",
    "FEATURE_QC_FAIL_REVIEW_SOURCE",
)
slot_features.to_csv(OUTPUT_DIR / "replacement_candidate_extracted_features.csv", index=False)
unique_sites.to_csv(OUTPUT_DIR / "unique_candidate_temporal_features.csv", index=False)
print("Candidate/slot output rows:", len(slot_features))
print(slot_features["candidate_status"].value_counts(dropna=False).to_string())

Candidate/slot output rows: 170
candidate_status
FEATURE_QC_FAIL_REVIEW_SOURCE                     127
FEATURE_QC_PASS_SPATIAL_AND_ASSIGNMENT_PENDING     43


## PART E — Reports, recovery gate and outputs
### 10. Per-slot and per-group feature availability

In [15]:
slot_summary = slot_features.groupby(
    ["matched_positive_id", "old_control_sample_id"], dropna=False
).agg(
    shortlisted_candidates=("sample_id", "size"),
    feature_qc_pass_candidates=("full_temporal_row_valid", "sum"),
    rain_valid_candidates=("rainfall_valid", "sum"),
    moisture_valid_candidates=("soil_moisture_complete", "sum"),
    strict_landcover_candidates=("landcover_valid_strict", "sum"),
    waterway_valid_candidates=("waterway_available", "sum"),
).reset_index()
slot_summary["has_provisional_feature_pass"] = slot_summary["feature_qc_pass_candidates"].gt(0)
slot_summary.to_csv(OUTPUT_DIR / "replacement_slot_feature_qc_summary.csv", index=False)

group_summary = slot_summary.groupby("matched_positive_id").agg(
    missing_control_slots=("old_control_sample_id", "size"),
    slots_with_feature_pass=("has_provisional_feature_pass", "sum"),
    total_feature_pass_options=("feature_qc_pass_candidates", "sum"),
).reset_index()
group_summary["all_slots_have_feature_pass_options"] = (
    group_summary["missing_control_slots"] == group_summary["slots_with_feature_pass"]
)
group_summary["verified_recovered"] = False
# Candidate collision, original spatial exclusion and control assignment still pending.
group_summary.to_csv(OUTPUT_DIR / "group_feature_recovery_readiness.csv", index=False)
print("Slots with at least one feature-QC pass:", int(slot_summary["has_provisional_feature_pass"].sum()),
      "/", len(slot_summary))
print("Groups with enough per-slot feature-QC options:",
      int(group_summary["all_slots_have_feature_pass_options"].sum()), "/", len(group_summary))
display(group_summary.head())

Slots with at least one feature-QC pass: 9 / 34
Groups with enough per-slot feature-QC options: 9 / 33


,matched_positive_id,missing_control_slots,slots_with_feature_pass,total_feature_pass_options,all_slots_have_feature_pass_options,verified_recovered
0,POS_0008,1,1,5,True,False
1,POS_0013,1,0,0,False,False
2,POS_0020,1,1,5,True,False
3,POS_0022,1,0,0,False,False
4,POS_0023,1,1,5,True,False


### 11. Decision gate and reproducibility metadata

In [16]:
gate = pd.DataFrame([
    {"check": "Manifest has 33 recovery groups", "status": "PASS" if manifest["matched_positive_id"].nunique() == 33 else "REVIEW", "value": manifest["matched_positive_id"].nunique()},
    {"check": "Manifest has 34 replacement slots", "status": "PASS" if len(slot_summary) == 34 else "REVIEW", "value": len(slot_summary)},
    {"check": "Extracted candidate/date observations", "status": "COMPLETED", "value": len(unique_sites)},
    {"check": "Unique candidates passing temporal row QC", "status": "DIAGNOSTIC", "value": int(unique_sites["full_temporal_row_valid"].sum())},
    {"check": "Replacement slots with >=1 feature-complete option", "status": "DIAGNOSTIC", "value": int(slot_summary["has_provisional_feature_pass"].sum())},
    {"check": "Groups with >=1 feature-complete option in every missing slot", "status": "DIAGNOSTIC", "value": int(group_summary["all_slots_have_feature_pass_options"].sum())},
    {"check": "Collection extraction errors", "status": "REVIEW" if climate_errors else "PASS", "value": len(climate_errors)},
    {"check": "Original pseudo-absence spatial exclusion validated", "status": "PENDING", "value": 0},
    {"check": "Candidate uniqueness and 1:2 matching validated", "status": "PENDING", "value": 0},
    {"check": "Independent pre-event annual landcover validated", "status": "LIMITATION", "value": 0},
    {"check": "Verified replacements approved", "status": "NOT_APPROVED", "value": 0},
    {"check": "Dataset rebuilding or retraining authorized", "status": "NO", "value": 0},
])
gate.to_csv(OUTPUT_DIR / "stage13b4_decision_gate.csv", index=False)

summary = {
    "input_manifest_rows": int(len(manifest)),
    "unique_candidate_date_pairs": int(len(unique_sites)),
    "unique_candidate_locations": int(unique_sites["sample_id"].nunique()),
    "candidate_date_rows_passing_original_feature_qc": int(unique_sites["full_temporal_row_valid"].sum()),
    "missing_control_slots": int(len(slot_summary)),
    "slots_with_at_least_one_feature_pass": int(slot_summary["has_provisional_feature_pass"].sum()),
    "groups_with_all_slots_having_feature_pass_options": int(group_summary["all_slots_have_feature_pass_options"].sum()),
    "earth_engine_extraction_errors": int(len(climate_errors)),
    "verified_recovered_groups": 0,
    "retraining_authorized": False,
    "leakage_caution": "Annual land-cover composites may include imagery from later in the event year",
}
with (OUTPUT_DIR / "stage13b4_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

display(gate)
print("\nSaved all Stage 13B.4 outputs to:", OUTPUT_DIR)

,check,status,value
0,Manifest has 33 recovery groups,PASS,33
1,Manifest has 34 replacement slots,PASS,34
2,Extracted candidate/date observations,COMPLETED,154
3,Unique candidates passing temporal row QC,DIAGNOSTIC,43
4,Replacement slots with >=1 feature-complete op...,DIAGNOSTIC,9
5,Groups with >=1 feature-complete option in eve...,DIAGNOSTIC,9
6,Collection extraction errors,PASS,0
7,Original pseudo-absence spatial exclusion vali...,PENDING,0
8,Candidate uniqueness and 1:2 matching validated,PENDING,0
9,Independent pre-event annual landcover validated,LIMITATION,0



Saved all Stage 13B.4 outputs to: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b4_historical_control_features


### Interpretation and what to upload

Upload these first:

1. `stage13b4_decision_gate.csv`
2. `stage13b4_summary.json`
3. `replacement_slot_feature_qc_summary.csv`
4. `group_feature_recovery_readiness.csv`
5. `replacement_candidate_extracted_features.csv`
6. `earth_engine_collection_counts.csv`
7. `earth_engine_extraction_errors.csv`

**Stage 13B.5** must still check the original pseudo-absence spatial-exclusion logic, assign unique replacements globally, preserve each group's historical reference date, and re-evaluate matched-group integrity. No output from this notebook is an authorized final recovered dataset. If Earth Engine authentication or access fails, resolve it first; missing outputs should not be interpreted as scientifically invalid candidates.

**Method caveat:** As in original Stage 05B, IMERG and ERA5 windows stop at the start of the recorded event day. Annual GLC/Dynamic World compositing has different time semantics; record limitations and investigate strict pre-event alternatives before asserting zero temporal leakage.